# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset using the `mlcroissant` library, referencing all record sets, fields, and columns by their `@id` identifiers.

### Dataset Source
The dataset is accessed and described via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, their fields, and the associated `@id` values in the dataset.

In [ ]:
# List all record sets by @id and their fields
record_sets = list(dataset.list_record_sets())
if not record_sets:
    print("No record sets defined in the Dataset Croissant. Check for updates or alternate schema.")
else:
    print("Available record sets (@id) and fields (@id):\n")
    for rs in record_sets:
        rs_metadata = dataset.get_record_set(rs)
        print(f"RecordSet @id: {rs_metadata['@id']}")
        fields = rs_metadata.get('field', [])
        if isinstance(fields, dict):  # Single field
            fields = [fields]
        for f in fields:
            if isinstance(f, str):
                print(f"  Field @id: {f}")
            elif isinstance(f, dict):
                print(f"  Field @id: {f.get('@id', '-')}")
        print("")

## 3. Data Extraction
Load data from all available record sets into DataFrames. Reference record sets and fields using their `@id` values.

In [ ]:
# Prepare to extract data for each record set (@id)
dataframes = {}
record_sets = list(dataset.list_record_sets())

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")
        else:
            print(f"No records loaded for RecordSet @id: {record_set_id}")
    except Exception as e:
        print(f"Error loading records for RecordSet @id: {record_set_id}: {e}")

# Display the columns/field @ids for each loaded DataFrame
for record_set_id, df in dataframes.items():
    print(f"\nRecordSet @id: {record_set_id}")
    print("Fields (@id):", list(df.columns))
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Demonstrate common analysis. We'll:
- Select a record set and numeric field (referenced by `@id`) if available
- Filter, normalize, and group data using `@id` references

In [ ]:
# --- Example EDA: Filter, normalize fields by @id ---
if dataframes:
    # Select the first available record set and try to locate a numeric field by type
    selected_record_set_id = list(dataframes.keys())[0]
    df = dataframes[selected_record_set_id]
    print(f"\nUsing RecordSet @id: {selected_record_set_id}")

    numeric_field_id = None
    # Try to infer a numeric field by column dtype
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    if numeric_field_id:
        print(f"Numeric field selected: {numeric_field_id}")
        # Drop missing values for analysis
        clean_df = df.dropna(subset=[numeric_field_id])
        threshold = clean_df[numeric_field_id].mean()

        filtered_df = clean_df[clean_df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a likely categorical field
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and pd.api.types.is_object_dtype(df[col]):
                group_field = col
                break
        if group_field:
            print(f"\nGrouping by field @id: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(grouped_df.head())
        else:
            print("No suitable categorical field found to group by.")
    else:
        print("No numeric fields available for EDA in this record set.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize distributions or relationships for selected fields using their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: no numeric field or dataframe loaded.")

## 6. Conclusion
In this notebook, you learned how to load and explore a dataset described by a Croissant schema with `mlcroissant`, referencing all data entities via their `@id` fields. Using this reproducible workflow, you can:
- Retrieve dataset metadata and inspect data structure
- Extract records from specific record sets using `@id`
- Perform basic exploratory analysis and visualizations referencing Croissant identifiers

Continue by applying custom preprocessing and more advanced analytics as needed!